# WBC White Blood Cell Classification
## Full Pipeline — Data Exploration + Preprocessing + Training + Submission

In [ ]:
!pip install timm albumentations --quiet

## 1. Imports

In [2]:
import os, random, warnings
import numpy as np
import pandas as pd
import cv2
from tqdm import tqdm
from PIL import Image
from scipy.optimize import minimize
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Sampler

from collections import defaultdict
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, classification_report

import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm

print('PyTorch :', torch.__version__)
print('Timm    :', timm.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))

PyTorch : 2.10.0+cu128
Timm    : 1.0.25
CUDA    : True
GPU     : Tesla T4


## 2. Configuration

In [3]:
CONFIG = {
    'img_size'    : 320,
    'batch_size'  : 16,
    'epochs'      : 40,
    'lr'          : 3e-4,
    'num_classes' : 13,
    'seed'        : 42,
    'num_workers' : 2,
    'label_smooth': 0.05,
    'n_tta'       : 10,
    'patience'    : 8,
}

BASE_PATH  = '/kaggle/input/competitions/ima205-challenge-2026/IMA205-challenge'
CACHE_DIR  = '/kaggle/input/datasets/ellefiala/wbc-cache/preprocessed'
WORK_DIR   = '/kaggle/working'
BEST_MODEL = f'{WORK_DIR}/best_convnext_small.pth'
SUB_PATH   = f'{WORK_DIR}/submission.csv'

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(CONFIG['seed'])
print('Config ready.')

Config ready.


## 3. Data Loading

In [4]:
train_df   = pd.read_csv(f'{BASE_PATH}/train_metadata.csv')
test_df    = pd.read_csv(f'{BASE_PATH}/test_metadata.csv')
TRAIN_IMGS = f'{BASE_PATH}/train'
TEST_IMGS  = f'{BASE_PATH}/test'

classes  = sorted(train_df['label'].unique())
label2id = {label: i for i, label in enumerate(classes)}
id2label = {i: label for label, i in label2id.items()}
train_df['label_id'] = train_df['label'].map(label2id)

# Confused pairs from confusion matrix — used by sampler and cost matrix
CONFUSED_PAIRS = [
    ('PLY', 'LY'),
    ('VLY', 'LY'),
    ('PC',  'PMY'),
    ('PC',  'LY'),
    ('MMY', 'BNE'),
    ('MMY', 'MY'),
    ('PMY', 'MY'),
]

# Ultra-rare: CutMix/Mixup disabled for these
ULTRA_RARE = ['PLY', 'PC']

print(f'Train: {len(train_df)} | Test: {len(test_df)}')
print(f'Classes: {classes}')
print('\nClass counts:')
print(train_df['label'].value_counts().to_string())

Train: 28901 | Test: 9634
Classes: ['BA', 'BL', 'BNE', 'EO', 'LY', 'MMY', 'MO', 'MY', 'PC', 'PLY', 'PMY', 'SNE', 'VLY']

Class counts:
label
SNE    13015
LY      8101
MO      2746
BL      2012
EO       861
MY       441
BA       415
BNE      391
VLY      366
MMY      360
PMY      114
PC        68
PLY       11


## 4. Train / Validation Split

In [5]:
train_data, val_data = train_test_split(
    train_df,
    test_size=0.2,
    stratify=train_df['label'],
    random_state=CONFIG['seed']
)
train_data = train_data.reset_index(drop=True)
val_data   = val_data.reset_index(drop=True)
print(f'Train: {len(train_data)} | Val: {len(val_data)}')

class_weights_np = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(CONFIG['num_classes']),
    y=train_data['label'].map(label2id).values
)
class_weights = torch.tensor(class_weights_np, dtype=torch.float).cuda()

print('\nClass weights:')
for label, idx in sorted(label2id.items(),
                          key=lambda x: -class_weights_np[x[1]]):
    n = (train_data['label'] == label).sum()
    print(f'  {label:4s}  w={class_weights_np[idx]:.3f}  n={n}')

Train: 23120 | Val: 5781

Class weights:
  PLY   w=197.607  n=9
  PC    w=32.934  n=54
  PMY   w=19.544  n=91
  MMY   w=6.175  n=288
  VLY   w=6.070  n=293
  BNE   w=5.682  n=313
  BA    w=5.357  n=332
  MY    w=5.038  n=353
  EO    w=2.581  n=689
  BL    w=1.105  n=1609
  MO    w=0.809  n=2197
  LY    w=0.274  n=6480
  SNE   w=0.171  n=10412


## 5. Data Augmentation

In [6]:
train_transforms = A.Compose([
    A.Resize(CONFIG['img_size'], CONFIG['img_size']),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Rotate(limit=45, p=0.8),
    A.ElasticTransform(alpha=1, sigma=10, p=0.3),
    A.GridDistortion(num_steps=5, distort_limit=0.2, p=0.3),
    A.RandomBrightnessContrast(brightness_limit=0.2,
                                contrast_limit=0.2, p=0.5),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=25,
                          val_shift_limit=10, p=0.4),
    A.RGBShift(r_shift_limit=15, g_shift_limit=15,
               b_shift_limit=15, p=0.3),
    A.RandomResizedCrop(size=(CONFIG['img_size'], CONFIG['img_size']),
                        scale=(0.70, 1.0), ratio=(0.9, 1.1), p=0.5),
    A.OneOf([
        A.GaussNoise(var_limit=(5.0, 25.0), p=1.0),
        A.GaussianBlur(blur_limit=(3, 5), p=1.0),
        A.Sharpen(alpha=(0.1, 0.3), p=1.0),
    ], p=0.3),
    A.CoarseDropout(max_holes=6, max_height=32, max_width=32,
                    min_holes=1, fill_value=0, p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_transforms = A.Compose([
    A.Resize(CONFIG['img_size'], CONFIG['img_size']),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

S = CONFIG['img_size']
def _norm():
    return [A.Normalize(mean=(0.485, 0.456, 0.406),
                        std=(0.229, 0.224, 0.225)), ToTensorV2()]

tta_transforms_list = [
    A.Compose([A.Resize(S, S)] + _norm()),
    A.Compose([A.Resize(S, S), A.HorizontalFlip(p=1)] + _norm()),
    A.Compose([A.Resize(S, S), A.VerticalFlip(p=1)] + _norm()),
    A.Compose([A.Resize(S, S), A.HorizontalFlip(p=1),
               A.VerticalFlip(p=1)] + _norm()),
    A.Compose([A.Resize(S, S), A.Rotate(limit=(90, 90), p=1)] + _norm()),
    A.Compose([A.Resize(S, S), A.Rotate(limit=(180, 180), p=1)] + _norm()),
    A.Compose([A.Resize(S, S), A.Rotate(limit=(270, 270), p=1)] + _norm()),
    A.Compose([A.Resize(int(S*1.1), int(S*1.1)),
               A.CenterCrop(S, S)] + _norm()),
    A.Compose([A.Resize(S, S), A.HorizontalFlip(p=1),
               A.Rotate(limit=(90, 90), p=1)] + _norm()),
    A.Compose([A.Resize(S, S),
               A.RandomBrightnessContrast(
                   brightness_limit=(0.1, 0.1),
                   contrast_limit=(0.1, 0.1), p=1)] + _norm()),
]
print(f'Augmentations ready. TTA passes: {len(tta_transforms_list)}')

Augmentations ready. TTA passes: 10


## 6. Dataset & DataLoaders

In [8]:
class WBCDataset(Dataset):
    def __init__(self, df, image_dir, transforms=None, has_labels=True):
        self.df         = df.reset_index(drop=True)
        self.image_dir  = image_dir
        self.transforms = transforms
        self.has_labels = has_labels
        self.cached     = (set(os.listdir(CACHE_DIR))
                           if os.path.exists(CACHE_DIR) else set())

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        # Load from cache (already denoised + stain normalised + CLAHE)
        # Fall back to raw dir only if not cached
        path = (os.path.join(CACHE_DIR, row['ID'])
                if row['ID'] in self.cached
                else os.path.join(self.image_dir, row['ID']))
        image = np.array(Image.open(path).convert('RGB'))
        if self.transforms:
            image = self.transforms(image=image)['image']
        if self.has_labels:
            return image, label2id[row['label']]
        return image

In [9]:
class ConfusionAwareSampler(Sampler):
    """
    Soft confusion-aware sampler.
    - 35% of batches: inject 1 sample from each side of ONE random
      confused pair (2 reserved slots), rest is weighted random.
    - 65% of batches: pure weighted random sampling.
    This preserves the natural data distribution while still
    periodically forcing the model to see confused pairs together.
    """
    def __init__(self, df, batch_size, label_col='label'):
        self.batch_size = batch_size
        self.labels     = df[label_col].values

        self.class_indices = defaultdict(list)
        for i, lbl in enumerate(self.labels):
            self.class_indices[lbl].append(i)

        # Only pairs that are actually in the dataset
        self.valid_pairs = [
            (a, b) for a, b in CONFUSED_PAIRS
            if self.class_indices[a] and self.class_indices[b]
        ]

        BOOST = {
            'PLY': 15.0, 'PC': 10.0, 'PMY': 6.0,
            'MMY': 4.0,  'VLY': 4.0,  'MY': 3.5,
            'BNE': 3.0,  'BA': 1.5,
        }
        counts = {lbl: len(idxs)
                  for lbl, idxs in self.class_indices.items()}
        self.weights = np.array([
            (1.0 / np.sqrt(max(counts[lbl], 1))) * BOOST.get(lbl, 1.0)
            for lbl in self.labels
        ])
        self.weights /= self.weights.sum()
        self.n_batches = len(df) // batch_size

    def __iter__(self):
        for _ in range(self.n_batches):
            # 35% of batches: inject one confused pair (2 slots)
            if random.random() < 0.35 and self.valid_pairs:
                cls_a, cls_b = random.choice(self.valid_pairs)
                guaranteed = [
                    random.choice(self.class_indices[cls_a]),
                    random.choice(self.class_indices[cls_b]),
                ]
                n_fill = self.batch_size - 2
            else:
                # 65% of batches: pure weighted random
                guaranteed = []
                n_fill = self.batch_size

            fill = np.random.choice(
                len(self.labels), size=n_fill,
                replace=True, p=self.weights
            ).tolist()

            batch = guaranteed + fill
            random.shuffle(batch)
            yield batch[:self.batch_size]

    def __len__(self):
        return self.n_batches


train_dataset = WBCDataset(train_data, TRAIN_IMGS, train_transforms)
val_dataset   = WBCDataset(val_data,   TRAIN_IMGS, val_transforms)
sampler       = ConfusionAwareSampler(train_data, CONFIG['batch_size'])

train_loader = DataLoader(train_dataset, batch_sampler=sampler,
                           num_workers=CONFIG['num_workers'], pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=CONFIG['batch_size'],
                           shuffle=False, num_workers=CONFIG['num_workers'],
                           pin_memory=True)
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

Train batches: 1445 | Val batches: 362


## 7. Loss Functions

In [10]:
class FocalLossWithCostMatrix(nn.Module):
    """
    Focal Loss + class weights + label smoothing + confusion cost matrix.

    The cost matrix adds an EXTRA penalty when the model confuses
    specific pairs identified in the confusion matrix analysis.
    For example: predicting LY when true label is PLY gets penalised
    3x more than a random mistake. This shapes the embedding space
    to push these pairs apart during training — a training-time fix,
    not a postprocessing band-aid.

    cost_matrix[i, j] = penalty multiplier when true=i, predicted=j
    Default = 1.0 everywhere, boosted for confused pairs.
    """
    def __init__(self, weight=None, gamma=3.0, label_smooth=0.05,
                 num_classes=13, cost_matrix=None):
        super().__init__()
        self.weight       = weight
        self.gamma        = gamma
        self.label_smooth = label_smooth
        self.num_classes  = num_classes
        # cost_matrix is a (C, C) tensor registered as buffer (not trained)
        if cost_matrix is not None:
            self.register_buffer('cost_matrix', cost_matrix)
        else:
            self.register_buffer('cost_matrix',
                                 torch.ones(num_classes, num_classes))

    def forward(self, inputs, targets):
        n = self.num_classes
        # Label smoothing
        with torch.no_grad():
            smooth = torch.zeros_like(inputs).scatter_(
                1, targets.unsqueeze(1), 1.0)
            smooth = smooth * (1 - self.label_smooth) + \
                     self.label_smooth / n

        log_p = F.log_softmax(inputs, dim=1)
        ce    = -(smooth * log_p).sum(dim=1)

        # Class weights
        if self.weight is not None:
            ce = ce * self.weight[targets]

        # Focal term
        pt     = torch.exp(-F.cross_entropy(inputs, targets,
                                             weight=self.weight,
                                             reduction='none'))
        focal  = ((1 - pt) ** self.gamma) * ce

        # Cost matrix: weight each sample by the cost of its predicted class
        # given the true class. If the model predicts a "costly" wrong class,
        # the gradient is amplified.
        preds      = inputs.argmax(dim=1)
        cost_weights = self.cost_matrix[targets, preds]
        focal      = focal * cost_weights

        return focal.mean()


class SupConLoss(nn.Module):
    """
    Supervised Contrastive Loss.
    Pulls same-class embeddings together, pushes different-class apart.
    Particularly effective for confused pairs: PLY/LY, MMY/MY, PC/PMY.
    Reference: Khosla et al., NeurIPS 2020.
    """
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        features = F.normalize(features, dim=1)
        sim      = torch.matmul(features, features.T) / self.temperature
        sim      = sim - sim.max(dim=1, keepdim=True)[0].detach()
        labels   = labels.view(-1, 1)
        mask     = torch.eq(labels, labels.T).float().to(features.device)
        eye      = torch.eye(len(labels), device=features.device)
        mask     = mask - eye
        exp_sim  = torch.exp(sim) * (1 - eye)
        log_prob = sim - torch.log(exp_sim.sum(dim=1, keepdim=True) + 1e-8)
        loss     = -(mask * log_prob).sum(dim=1) / (mask.sum(dim=1) + 1e-8)
        return loss.mean()


# ── Build the cost matrix ─────────────────────────────────────────────────
# cost_matrix[true_class, predicted_class]
# Default = 1.0 (normal penalty)
# Confused pairs get a higher multiplier so the model learns harder
# to separate them. Values chosen based on confusion matrix severity.
CONFUSED_COSTS = {
    ('PLY', 'LY') : 4.0,   # PLY→LY: worst confusion (50% misrate)
    ('LY',  'PLY'): 4.0,   # symmetric
    ('PC',  'PMY'): 3.5,   # PC→PMY: 36% misrate
    ('PMY', 'PC') : 3.5,
    ('PC',  'LY') : 3.0,   # PC→LY: 14% misrate
    ('LY',  'PC') : 3.0,
    ('VLY', 'LY') : 3.0,   # VLY→LY: 14% misrate
    ('LY',  'VLY'): 3.0,
    ('MMY', 'BNE'): 2.5,   # MMY→BNE: 17% misrate
    ('BNE', 'MMY'): 2.5,
    ('MMY', 'MY') : 2.5,   # MMY→MY: 15% misrate
    ('MY',  'MMY'): 2.5,
    ('PMY', 'MY') : 2.0,   # PMY→MY: 22% misrate
    ('MY',  'PMY'): 2.0,
    ('PLY', 'PC') : 2.0,   # secondary confusions
    ('PC',  'PLY'): 2.0,
}

cost_matrix = torch.ones(CONFIG['num_classes'], CONFIG['num_classes'])
for (true_lbl, pred_lbl), cost in CONFUSED_COSTS.items():
    if true_lbl in label2id and pred_lbl in label2id:
        i = label2id[true_lbl]
        j = label2id[pred_lbl]
        cost_matrix[i, j] = cost

# Diagonal stays 1.0 (correct predictions not penalised)
for i in range(CONFIG['num_classes']):
    cost_matrix[i, i] = 1.0

cost_matrix = cost_matrix.cuda()

focal_loss  = FocalLossWithCostMatrix(
    weight=class_weights, gamma=3.0,
    label_smooth=CONFIG['label_smooth'],
    num_classes=CONFIG['num_classes'],
    cost_matrix=cost_matrix
)
supcon_loss = SupConLoss(temperature=0.07)

print('Cost matrix:')
for (tl, pl), c in sorted(CONFUSED_COSTS.items(), key=lambda x: -x[1]):
    print(f'  {tl}→{pl}: {c}x penalty')

Cost matrix:
  PLY→LY: 4.0x penalty
  LY→PLY: 4.0x penalty
  PC→PMY: 3.5x penalty
  PMY→PC: 3.5x penalty
  PC→LY: 3.0x penalty
  LY→PC: 3.0x penalty
  VLY→LY: 3.0x penalty
  LY→VLY: 3.0x penalty
  MMY→BNE: 2.5x penalty
  BNE→MMY: 2.5x penalty
  MMY→MY: 2.5x penalty
  MY→MMY: 2.5x penalty
  PMY→MY: 2.0x penalty
  MY→PMY: 2.0x penalty
  PLY→PC: 2.0x penalty
  PC→PLY: 2.0x penalty


In [11]:
ULTRA_RARE_IDS = {label2id[c] for c in ULTRA_RARE}

def should_mix(labels_a, labels_b):
    """Only mix if neither batch contains ultra-rare samples (PLY, PC).
    Mixing 11 rare samples destroys signal instead of amplifying it."""
    a_ids = set(labels_a.cpu().numpy().tolist())
    b_ids = set(labels_b.cpu().numpy().tolist())
    return (len(a_ids & ULTRA_RARE_IDS) == 0 and
            len(b_ids & ULTRA_RARE_IDS) == 0)

def mixup_batch(images, labels, alpha=0.4):
    lam   = np.random.beta(alpha, alpha)
    idx = torch.randperm(images.size(0)).to(images.device)
    mixed = lam * images + (1 - lam) * images[idx]
    return mixed, labels, labels[idx], lam

def cutmix_batch(images, labels, alpha=1.0):
    lam   = np.random.beta(alpha, alpha)
    idx = torch.randperm(images.size(0)).to(images.device)
    W, H  = images.size(2), images.size(3)
    cut_w = int(W * np.sqrt(1 - lam))
    cut_h = int(H * np.sqrt(1 - lam))
    cx, cy = np.random.randint(W), np.random.randint(H)
    x1, x2 = max(0, cx - cut_w//2), min(W, cx + cut_w//2)
    y1, y2 = max(0, cy - cut_h//2), min(H, cy + cut_h//2)
    mixed  = images.clone()
    mixed[:, :, x1:x2, y1:y2] = images[idx, :, x1:x2, y1:y2]
    lam    = 1 - (x2-x1)*(y2-y1) / (W*H)
    return mixed, labels, labels[idx], lam

print('Mixup + CutMix defined.')

Mixup + CutMix defined.


## 8. Training Augmentation Helpers

In [12]:
class WBCModel(nn.Module):
    def __init__(self, model_name, num_classes, dropout=0.4):
        super().__init__()
        self.backbone = timm.create_model(
            model_name, pretrained=True,
            num_classes=0, global_pool='avg')
        in_features = self.backbone.num_features
        self.projector = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Linear(256, 128)
        )
        self.head = nn.Sequential(
            nn.BatchNorm1d(in_features),
            nn.Dropout(dropout),
            nn.Linear(in_features, 512),
            nn.GELU(),
            nn.BatchNorm1d(512),
            nn.Dropout(dropout * 0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x, return_features=False):
        features = self.backbone(x)
        logits   = self.head(features)
        if return_features:
            return logits, self.projector(features)
        return logits

model = WBCModel('convnext_small.fb_in22k_ft_in1k',
                  CONFIG['num_classes']).cuda()
total_p = sum(p.numel() for p in model.parameters()) / 1e6
print(f'ConvNeXt-Small — {total_p:.1f}M params')

model.safetensors:   0%|          | 0.00/201M [00:00<?, ?B/s]

ConvNeXt-Small — 50.1M params


In [13]:
def get_layerwise_param_groups(model, base_lr):
    """
    head          : base_lr       (learns fastest)
    backbone_late : base_lr / 10  (stages 2-3)
    backbone_early: base_lr / 100 (stages 0-1, nearly frozen)
    Prevents catastrophic forgetting of ImageNet-22k features.
    """
    head_params  = (list(model.head.parameters()) +
                    list(model.projector.parameters()))
    late_params, early_params = [], []
    for name, p in model.backbone.named_parameters():
        if any(s in name for s in ['stages.2', 'stages.3', 'norm']):
            late_params.append(p)
        else:
            early_params.append(p)
    return [
        {'params': head_params,  'lr': base_lr,        'name': 'head'},
        {'params': late_params,  'lr': base_lr / 10,   'name': 'bb_late'},
        {'params': early_params, 'lr': base_lr / 100,  'name': 'bb_early'},
    ]


def train_one_epoch(model, loader, optimizer, scheduler,
                    supcon_weight=0.05, mixup_prob=0.3, cutmix_prob=0.3):
    model.train()
    total_loss, all_preds, all_labels = 0.0, [], []
    for images, labels in tqdm(loader, desc='Train', leave=False):
        images, labels = images.cuda(), labels.cuda()
        optimizer.zero_grad()
        r       = random.random()
        idx_p = torch.randperm(images.size(0)).to(images.device)
        if r < cutmix_prob and should_mix(labels, labels[idx_p]):
            images, la, lb, lam = cutmix_batch(images, labels)
            logits, _ = model(images, return_features=True)
            loss = lam * focal_loss(logits, la) + (1-lam) * focal_loss(logits, lb)
        elif r < cutmix_prob + mixup_prob and should_mix(labels, labels[idx_p]):
            images, la, lb, lam = mixup_batch(images, labels)
            logits, _ = model(images, return_features=True)
            loss = lam * focal_loss(logits, la) + (1-lam) * focal_loss(logits, lb)
        else:
            logits, proj = model(images, return_features=True)
            loss = (focal_loss(logits, labels) +
                    supcon_weight * supcon_loss(proj, labels))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        if scheduler is not None:
            scheduler.step()
        total_loss += loss.item()
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return total_loss / len(loader), f1


@torch.no_grad()
def validate(model, loader):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    for images, labels in tqdm(loader, desc='Val', leave=False):
        logits = model(images.cuda())
        probs  = torch.softmax(logits, dim=1).cpu().numpy()
        all_probs.extend(probs)
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())
    f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return f1, np.array(all_labels), np.array(all_preds), np.array(all_probs)

print('Training functions defined.')

Training functions defined.


## 9. Training Loop

In [ ]:
for p in model.backbone.parameters():
    p.requires_grad = False

opt_p1 = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CONFIG['lr'], weight_decay=1e-4)
sch_p1 = optim.lr_scheduler.OneCycleLR(
    opt_p1, max_lr=CONFIG['lr'],
    steps_per_epoch=len(train_loader),
    epochs=5, pct_start=0.3, anneal_strategy='cos')

print('Phase 1 — warmup head (5 epochs, backbone frozen)...')
best_f1 = 0.0
history = {'train_loss': [], 'train_f1': [], 'val_f1': []}

for epoch in range(5):
    tr_loss, tr_f1 = train_one_epoch(
        model, train_loader, opt_p1, sch_p1,
        supcon_weight=0.0, mixup_prob=0.0, cutmix_prob=0.0)
    val_f1, val_labels, val_preds, val_probs = validate(model, val_loader)
    history['train_loss'].append(tr_loss)
    history['train_f1'].append(tr_f1)
    history['val_f1'].append(val_f1)
    print(f'  [Warmup {epoch+1}/5] loss={tr_loss:.4f} '
          f'train_f1={tr_f1:.4f} val_f1={val_f1:.4f}')
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), BEST_MODEL)
        print(f'  *** Saved — val_f1={best_f1:.4f}')

In [ ]:
for p in model.backbone.parameters():
    p.requires_grad = True

base_lr    = CONFIG['lr'] / 5
param_grps = get_layerwise_param_groups(model, base_lr)
opt_p2     = optim.AdamW(param_grps, weight_decay=1e-4)
sch_p2     = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    opt_p2, T_0=10, T_mult=2, eta_min=1e-7)

print('Phase 2 — full fine-tune with layer-wise LR decay...')
print(f'  head={base_lr:.2e} | late={base_lr/10:.2e} | early={base_lr/100:.2e}')

no_improve = 0
for epoch in range(CONFIG['epochs']):
    tr_loss, tr_f1 = train_one_epoch(
        model, train_loader, opt_p2, None,
        supcon_weight=0.05, mixup_prob=0.3, cutmix_prob=0.3)
    sch_p2.step()
    val_f1, val_labels, val_preds, val_probs = validate(model, val_loader)
    history['train_loss'].append(tr_loss)
    history['train_f1'].append(tr_f1)
    history['val_f1'].append(val_f1)
    lr_now = opt_p2.param_groups[0]['lr']
    print(f'  [Epoch {epoch+1:2d}/{CONFIG["epochs"]}] '
          f'loss={tr_loss:.4f} train_f1={tr_f1:.4f} '
          f'val_f1={val_f1:.4f} lr={lr_now:.2e}')
    if val_f1 > best_f1:
        best_f1    = val_f1
        no_improve = 0
        torch.save(model.state_dict(), BEST_MODEL)
        print(f'  *** Best saved — val_f1={best_f1:.4f}')
    else:
        no_improve += 1
        if no_improve >= CONFIG['patience']:
            print(f'  Early stop at epoch {epoch+1}')
            break

print(f'\nTraining done. Best val F1: {best_f1:.4f}')

In [ ]:
model.load_state_dict(torch.load(BEST_MODEL))
val_f1, val_labels, val_preds, val_probs = validate(model, val_loader)

print(f'Best model val F1: {val_f1:.4f}\n')
print(classification_report(val_labels, val_preds,
                              target_names=classes, zero_division=0))
f1_per_class = f1_score(val_labels, val_preds, average=None, zero_division=0)
print('\nPer-class F1:')
for cls, score in zip(classes, f1_per_class):
    bar = '█' * int(score * 20)
    print(f'  {cls:4s} {score:.3f} {bar}')

## 10. Post-processing — Temperature Scaling + Threshold Optimisation

In [ ]:
class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.temperature = nn.Parameter(torch.ones(1) * 1.5)
    def forward(self, logits):
        return logits / self.temperature

@torch.no_grad()
def get_logits(model, loader):
    model.eval()
    all_logits, all_labels = [], []
    for images, labels in tqdm(loader, desc='Collecting logits', leave=False):
        all_logits.append(model(images.cuda()).cpu())
        all_labels.append(labels)
    return torch.cat(all_logits), torch.cat(all_labels)

print('Collecting validation logits...')
val_logits, val_labels_ts = get_logits(model, val_loader)

temp_scaler  = TemperatureScaler()
opt_ts       = optim.LBFGS([temp_scaler.temperature], lr=0.01, max_iter=200)
nll_loss     = nn.CrossEntropyLoss()

def eval_temp():
    opt_ts.zero_grad()
    loss = nll_loss(temp_scaler(val_logits), val_labels_ts)
    loss.backward()
    return loss

opt_ts.step(eval_temp)
T_opt = temp_scaler.temperature.item()
print(f'Optimal temperature T = {T_opt:.4f}')
print('(T > 1 means model was overconfident on easy classes)')

## 11. TTA Inference & Submission

In [ ]:
def apply_temperature(probs_np, T):
    """Re-apply temperature to probability array via log-space rescaling."""
    logits = np.log(probs_np + 1e-10)
    scaled = logits / T
    scaled -= scaled.max(axis=1, keepdims=True)
    exp    = np.exp(scaled)
    return exp / exp.sum(axis=1, keepdims=True)

def apply_thresholds(probs, thresholds):
    return (probs * thresholds).argmax(axis=1)

def neg_f1(log_thresh, probs, labels):
    preds = apply_thresholds(probs, np.exp(log_thresh))
    return -f1_score(labels, preds, average='macro', zero_division=0)

# Apply temperature scaling first
val_probs_ts = apply_temperature(val_probs, T_opt)

f1_before = f1_score(val_labels,
                      val_probs_ts.argmax(axis=1),
                      average='macro', zero_division=0)
print(f'F1 after temperature scaling   : {f1_before:.4f}')

# Initial thresholds: start with a boost for rare/confused classes
BOOST_INIT = {
    'PLY': 2.0, 'PC': 1.8, 'PMY': 1.5,
    'MMY': 1.4, 'VLY': 1.4, 'MY': 1.3, 'BNE': 1.2,
}
init_thresh = np.array([
    BOOST_INIT.get(id2label[i], 1.0)
    for i in range(CONFIG['num_classes'])
])

result = minimize(
    neg_f1,
    x0=np.log(init_thresh),
    args=(val_probs_ts, val_labels),
    method='Nelder-Mead',
    options={'maxiter': 3000, 'xatol': 1e-6, 'fatol': 1e-6}
)

opt_thresh = np.exp(result.x)
f1_after   = f1_score(val_labels,
                       apply_thresholds(val_probs_ts, opt_thresh),
                       average='macro', zero_division=0)

print(f'F1 after threshold calibration : {f1_after:.4f}  '
      f'(gain = +{f1_after - f1_before:.4f})')
print('\nOptimised thresholds:')
for i in range(CONFIG['num_classes']):
    flag = ' ◄' if opt_thresh[i] > 1.1 else ''
    print(f'  {id2label[i]:4s}: {opt_thresh[i]:.4f}{flag}')

In [ ]:
model.load_state_dict(torch.load(BEST_MODEL))
model.eval()

print(f'Running {len(tta_transforms_list)}-pass TTA...')
all_tta_probs = []

for i, aug in enumerate(tta_transforms_list):
    ds = WBCDataset(test_df, TEST_IMGS, aug,
                    has_labels=False)
    dl = DataLoader(ds, batch_size=32, shuffle=False,
                    num_workers=CONFIG['num_workers'], pin_memory=True)
    probs = []
    with torch.no_grad():
        for images in tqdm(dl, desc=f'TTA {i+1}/{len(tta_transforms_list)}',
                           leave=False):
            probs.append(torch.softmax(
                model(images.cuda()), dim=1).cpu().numpy())
    all_tta_probs.append(np.concatenate(probs, axis=0))
    print(f'  Pass {i+1} done.')

# Average TTA → temperature scaling → threshold calibration
mean_probs     = np.mean(all_tta_probs, axis=0)
test_probs_ts  = apply_temperature(mean_probs, T_opt)
final_preds    = apply_thresholds(test_probs_ts, opt_thresh)
pred_labels    = [id2label[i] for i in final_preds]

# Save submission
sub = pd.DataFrame({'ID': test_df['ID'].values, 'label': pred_labels})
sub.to_csv(SUB_PATH, index=False)

assert len(sub) == len(test_df)
assert set(sub['label'].unique()).issubset(set(classes))

print(f'\nSubmission saved: {SUB_PATH}')
print(f'Rows: {len(sub)} | Unique labels: {sub["label"].nunique()}')
print()
print('── Val F1 summary ──────────────────────────────')
print(f'  Raw model                 : {val_f1:.4f}')
print(f'  + Temperature scaling     : {f1_before:.4f}')
print(f'  + Threshold calibration   : {f1_after:.4f}')
print(f'  Total postprocessing gain : +{f1_after - val_f1:.4f}')
print()
print('Test prediction distribution:')
print(pd.Series(pred_labels).value_counts().to_string())
display(sub.head(10))